# ⚙️ DQ Framework — Notebook 2: Rule Engine

**Purpose:** Defines the `RuleEngine` class with all 16 rule type implementations.
This notebook is `%run` by the controller and depends on nothing except PySpark.

### Rule types implemented
| # | Type | Description |
|---|---|---|
| 1 | `not_null` | Column must have zero nulls |
| 2 | `unique` | Column combination must be unique |
| 3 | `row_count` | Row count within bounds |
| 4 | `accepted_values` | Column values from an allowed list |
| 5 | `regex` | Values match a regex pattern |
| 6 | `range` | Numeric column within bounds |
| 7 | `date_range` | Date column within bounds |
| 8 | `referential` | FK values exist in reference table |
| 9 | `freshness` | Data updated within N hours |
| 10 | `custom_sql` | Arbitrary SQL expression |
| 11 | `completeness` | Null rate below threshold |
| 12 | `duplicate_count` | Duplicate count below threshold |
| 13 | `conditional_not_null` | Not-null only when a condition is met |
| 14 | `mutual_exclusivity` | Exactly one column in a group populated |
| 15 | `character_set` | Text contains only allowed characters |
| 16 | `length_check` | String length within bounds |

## Imports

In [0]:
import re
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F

## DQRuleResult — Result Container

In [0]:
@dataclass
class DQRuleResult:
    """
    Holds the outcome of a single DQ rule execution.

    Attributes
    ----------
    dataset_name  : dataset this result belongs to
    rule_id       : unique rule identifier
    rule_type     : rule type string (e.g. 'not_null')
    description   : human-readable rule description
    severity      : CRITICAL | WARNING | INFO
    passed        : True if the rule passed
    row_count     : number of failing rows (0 for rules like freshness/row_count)
    details       : human-readable detail message
    error         : non-empty if the rule itself threw an exception
    run_timestamp : UTC ISO timestamp of this result
    """
    dataset_name:  str
    rule_id:       str
    rule_type:     str
    description:   str
    severity:      str
    passed:        bool
    row_count:     int  = 0
    details:       str  = ""
    error:         str  = ""
    run_timestamp: str  = field(default_factory=lambda: datetime.utcnow().isoformat())

    @property
    def status(self):
        if self.error:
            return "ERROR"
        return "PASS" if self.passed else "FAIL"

## RuleEngine — All 16 Rule Implementations

In [0]:
class RuleEngine:
    """
    Executes DQ rule dicts against a Spark DataFrame.

    Usage
    -----
    engine = RuleEngine(spark)
    result = engine.execute(rule_dict, dataframe, "my_dataset")
    """

    def __init__(self, spark_session):
        self.spark = spark_session

    # ─────────────────────────────────────────────────────────────────────────
    # Public — dispatcher
    # ─────────────────────────────────────────────────────────────────────────
    def execute(self, rule, df, dataset_name):
        """
        Execute one rule against df.  Returns a DQRuleResult — never raises.
        Any unexpected exception is captured in result.error.
        """
        rule_type = rule.get("type", "unknown")
        base = dict(
            dataset_name = dataset_name,
            rule_id      = rule.get("id", "unknown"),
            rule_type    = rule_type,
            description  = rule.get("description", ""),
            severity     = rule.get("severity", "WARNING"),
        )
        try:
            handler = getattr(self, f"_rule_{rule_type}", None)
            if handler is None:
                return DQRuleResult(**base, passed=False,
                                   error=f"Unknown rule type '{rule_type}'. "
                                         f"Check VALID_RULE_TYPES in notebook 01.")
            return handler(rule, df, base)
        except Exception as exc:
            return DQRuleResult(**base, passed=False,
                                error=f"Rule execution error: {exc}")

    # ─────────────────────────────────────────────────────────────────────────
    # Rules 1–12  (original)
    # ─────────────────────────────────────────────────────────────────────────

    def _rule_not_null(self, rule, df, base):
        """Column must have zero null values."""
        col        = rule["column"]
        null_count = df.filter(F.col(col).isNull()).count()
        passed     = null_count == 0
        return DQRuleResult(
            **base, passed=passed, row_count=null_count,
            details=(
                f"'{col}' has {null_count} null row(s)."
                if not passed else
                f"'{col}' — no nulls found. ✓"
            ),
        )

    def _rule_unique(self, rule, df, base):
        """Column combination must be unique across all rows."""
        cols     = rule["columns"]
        total    = df.count()
        distinct = df.select(*cols).distinct().count()
        dups     = total - distinct
        passed   = dups == 0
        return DQRuleResult(
            **base, passed=passed, row_count=dups,
            details=(
                f"{dups} duplicate row(s) on {cols} (total={total}, distinct={distinct})."
                if not passed else
                f"All {total} rows are unique on {cols}. ✓"
            ),
        )

    def _rule_row_count(self, rule, df, base):
        """Total row count must be within [min, max]."""
        count  = df.count()
        mn, mx = rule.get("min"), rule.get("max")
        fails  = []
        if mn is not None and count < mn:
            fails.append(f"count {count:,} < min {mn:,}")
        if mx is not None and count > mx:
            fails.append(f"count {count:,} > max {mx:,}")
        passed = not fails
        return DQRuleResult(
            **base, passed=passed, row_count=count,
            details=(
                "; ".join(fails)
                if not passed else
                f"Row count {count:,} is within bounds [{mn}, {mx}]. ✓"
            ),
        )

    def _rule_accepted_values(self, rule, df, base):
        """Column values must be from an allowed list."""
        col    = rule["column"]
        values = [str(v) for v in rule["values"]]
        bad_df = df.filter(
            F.col(col).isNotNull() & ~F.col(col).cast("string").isin(values)
        )
        bad_count = bad_df.count()
        passed    = bad_count == 0
        sample    = []
        if not passed:
            sample = [r[col] for r in bad_df.select(col).distinct().limit(5).collect()]
        return DQRuleResult(
            **base, passed=passed, row_count=bad_count,
            details=(
                f"{bad_count} row(s) in '{col}' have invalid values. Sample: {sample}"
                if not passed else
                f"All non-null '{col}' values are in the accepted list. ✓"
            ),
        )

    def _rule_regex(self, rule, df, base):
        """Column values must match a regex pattern."""
        col     = rule["column"]
        pattern = rule["pattern"]
        bad_count = df.filter(
            F.col(col).isNotNull() & ~F.col(col).cast("string").rlike(pattern)
        ).count()
        passed = bad_count == 0
        return DQRuleResult(
            **base, passed=passed, row_count=bad_count,
            details=(
                f"{bad_count} non-null row(s) in '{col}' do not match pattern '{pattern}'."
                if not passed else
                f"All non-null '{col}' values match the pattern. ✓"
            ),
        )

    def _rule_range(self, rule, df, base):
        """Numeric column must be within [min, max]."""
        col  = rule["column"]
        mn   = rule.get("min")
        mx   = rule.get("max")
        cond = F.lit(False)
        if mn is not None:
            cond = cond | (F.col(col) < mn)
        if mx is not None:
            cond = cond | (F.col(col) > mx)
        bad_count = df.filter(F.col(col).isNotNull() & cond).count()
        passed    = bad_count == 0
        return DQRuleResult(
            **base, passed=passed, row_count=bad_count,
            details=(
                f"{bad_count} row(s) in '{col}' are outside range [{mn}, {mx}]."
                if not passed else
                f"All non-null '{col}' values are within [{mn}, {mx}]. ✓"
            ),
        )

    def _rule_date_range(self, rule, df, base):
        """Date column must be within [min, max]. Use 'today' as max for current date."""
        col     = rule["column"]
        min_val = rule.get("min")
        max_val = rule.get("max")
        today   = datetime.now(timezone.utc).strftime("%Y-%m-%d")
        if max_val == "today":
            max_val = today
        cond = F.lit(False)
        if min_val:
            cond = cond | (F.col(col).cast("date") < F.lit(min_val).cast("date"))
        if max_val:
            cond = cond | (F.col(col).cast("date") > F.lit(max_val).cast("date"))
        bad_count = df.filter(F.col(col).isNotNull() & cond).count()
        passed    = bad_count == 0
        return DQRuleResult(
            **base, passed=passed, row_count=bad_count,
            details=(
                f"{bad_count} row(s) in '{col}' are outside date range [{min_val}, {max_val}]."
                if not passed else
                f"All non-null dates in '{col}' are within [{min_val}, {max_val}]. ✓"
            ),
        )

    def _rule_referential(self, rule, df, base):
        """FK column values must exist in a reference table column."""
        col       = rule["column"]
        ref_table = rule["ref_table"]
        ref_col   = rule["ref_column"]
        ref_df    = (
            self.spark.table(ref_table)
            .select(F.col(ref_col).alias("__ref__"))
            .distinct()
        )
        bad_count = df.join(ref_df, df[col] == ref_df["__ref__"], "left_anti").count()
        passed    = bad_count == 0
        return DQRuleResult(
            **base, passed=passed, row_count=bad_count,
            details=(
                f"{bad_count} row(s) in '{col}' not found in {ref_table}.{ref_col}."
                if not passed else
                f"All '{col}' values exist in {ref_table}.{ref_col}. ✓"
            ),
        )

    def _rule_freshness(self, rule, df, base):
        """Data must have been updated within max_hours of now."""
        col       = rule["column"]
        max_hours = float(rule["max_hours"])
        threshold = datetime.now(timezone.utc) - timedelta(hours=max_hours)

        latest_row = df.agg(F.max(F.col(col)).alias("latest")).collect()[0]
        latest     = latest_row["latest"]

        if latest is None:
            return DQRuleResult(
                **base, passed=False, row_count=0,
                details=f"Column '{col}' has no non-null values — freshness cannot be checked.",
            )

        # Normalise to timezone-aware datetime
        if hasattr(latest, "tzinfo") and latest.tzinfo is None:
            latest = latest.replace(tzinfo=timezone.utc)
        elif not hasattr(latest, "hour"):                # date object, not datetime
            latest = datetime(latest.year, latest.month, latest.day, tzinfo=timezone.utc)

        passed  = latest >= threshold
        age_hrs = (datetime.now(timezone.utc) - latest).total_seconds() / 3600
        return DQRuleResult(
            **base, passed=passed, row_count=0,
            details=(
                f"Latest '{col}' = {latest.isoformat()} ({age_hrs:.1f}h ago). "
                f"Max allowed: {max_hours}h. {'⚠ STALE' if not passed else '✓ FRESH'}."
            ),
        )

    def _rule_custom_sql(self, rule, df, base):
        """
        Arbitrary SQL expression.
        The SQL must return exactly two columns: pass (BOOLEAN), row_count (LONG).
        Use {table} as a placeholder for the dataset — it will be replaced with a
        temp view name at runtime.
        """
        sql_template = rule["sql"]
        view_name    = f"__dq_{base['dataset_name']}_{base['rule_id']}__".replace("-", "_")
        df.createOrReplaceTempView(view_name)
        resolved_sql = sql_template.replace("{table}", view_name)
        row          = self.spark.sql(resolved_sql).collect()[0]
        passed       = bool(row["pass"])
        row_count    = int(row.get("row_count", 0))
        return DQRuleResult(
            **base, passed=passed, row_count=row_count,
            details=f"Custom SQL → pass={passed}, failing_rows={row_count}.",
        )

    def _rule_completeness(self, rule, df, base):
        """Null rate of a column must be at or below max_null_rate (0.0–1.0)."""
        col           = rule["column"]
        max_null_rate = float(rule["max_null_rate"])
        total         = df.count()
        if total == 0:
            return DQRuleResult(**base, passed=True, row_count=0,
                                details="Table is empty — completeness check skipped.")
        null_count = df.filter(F.col(col).isNull()).count()
        null_rate  = null_count / total
        passed     = null_rate <= max_null_rate
        return DQRuleResult(
            **base, passed=passed, row_count=null_count,
            details=(
                f"'{col}' null rate: {null_rate:.2%} ({null_count:,}/{total:,} rows). "
                f"Max allowed: {max_null_rate:.2%}."
                if not passed else
                f"'{col}' null rate {null_rate:.2%} is within the {max_null_rate:.2%} threshold. ✓"
            ),
        )

    def _rule_duplicate_count(self, rule, df, base):
        """Number of duplicate row groups must be at or below max_duplicates."""
        cols     = rule["columns"]
        max_dups = int(rule.get("max_duplicates", 0))
        dup_count = (
            df.groupBy(*cols)
              .count()
              .filter(F.col("count") > 1)
              .count()
        )
        passed = dup_count <= max_dups
        return DQRuleResult(
            **base, passed=passed, row_count=dup_count,
            details=(
                f"Duplicate groups on {cols}: {dup_count} (max allowed: {max_dups})."
                if not passed else
                f"Duplicate groups on {cols}: {dup_count} — within the {max_dups} threshold. ✓"
            ),
        )

    # ─────────────────────────────────────────────────────────────────────────
    # Rules 13–16  (new)
    # ─────────────────────────────────────────────────────────────────────────

    def _rule_conditional_not_null(self, rule, df, base):
        """
        Enforces that 'column' is NOT NULL only when a specific condition on
        'condition_column' is satisfied.
        """
        col      = rule["column"]
        cond_col = rule["condition_column"]
        cond_val = rule.get("condition_value")
        op       = rule.get("condition_operator", "eq").lower()

        trigger   = self._build_condition(cond_col, op, cond_val)
        scope     = df.filter(trigger).count()
        bad_count = df.filter(trigger & F.col(col).isNull()).count()
        passed    = bad_count == 0

        op_label = self._op_display(op, cond_col, cond_val)
        return DQRuleResult(
            **base, passed=passed, row_count=bad_count,
            details=(
                f"{bad_count} row(s) where ({op_label}) have '{col}' = NULL. "
                f"Scope: {scope} row(s) matched the condition."
                if not passed else
                f"'{col}' is non-null in all {scope} row(s) where ({op_label}). ✓"
            ),
        )

    def _rule_mutual_exclusivity(self, rule, df, base):
        """
        Ensures exactly ONE column in a group is populated (non-null) per row.
        All other columns in the group must be null.
        """
        cols           = rule["columns"]
        allow_all_null = rule.get("allow_all_null", False)

        # Count non-null columns per row
        non_null_expr = sum(
            F.when(F.col(c).isNotNull(), F.lit(1)).otherwise(F.lit(0))
            for c in cols
        )
        df_with_cnt = df.withColumn("__non_null_cnt__", non_null_expr)

        if allow_all_null:
            bad_df = df_with_cnt.filter(~F.col("__non_null_cnt__").isin(0, 1))
        else:
            bad_df = df_with_cnt.filter(F.col("__non_null_cnt__") != 1)

        bad_count = bad_df.count()
        total     = df.count()
        passed    = bad_count == 0

        breakdown = (
            df_with_cnt
            .groupBy("__non_null_cnt__")
            .count()
            .orderBy("__non_null_cnt__")
            .collect()
        )
        bd_str = ", ".join(
            f"{r['__non_null_cnt__']} col(s) set → {r['count']} row(s)"
            for r in breakdown
        )

        return DQRuleResult(
            **base, passed=passed, row_count=bad_count,
            details=(
                f"{bad_count}/{total} row(s) violate mutual exclusivity on {cols}. "
                f"Breakdown: [{bd_str}]. "
                f"({'all-null rows allowed' if allow_all_null else 'all-null rows are violations'})"
                if not passed else
                f"All {total} row(s) satisfy mutual exclusivity on {cols}. "
                f"Breakdown: [{bd_str}]. ✓"
            ),
        )

    def _rule_character_set(self, rule, df, base):
        """
        Validates that a text column contains ONLY characters from an allowed set.
        Uses charset_name (built-in) or allowed_chars (explicit string).
        """
        col         = rule["column"]
        allow_empty = rule.get("allow_empty", True)

        if "charset_name" in rule:
            charset_name = rule["charset_name"]
            pattern      = CHARSET_PATTERNS.get(charset_name)
            if pattern is None:
                return DQRuleResult(
                    **base, passed=False,
                    error=f"Unknown charset_name '{charset_name}'. "
                          f"Allowed: {sorted(CHARSET_PATTERNS)}."
                )
            pattern_desc = f"charset '{charset_name}'"
        else:
            allowed_chars = rule["allowed_chars"]
            escaped       = re.escape(allowed_chars)
            pattern       = f"^[{escaped}]*$"
            preview       = allowed_chars[:40] + ("..." if len(allowed_chars) > 40 else "")
            pattern_desc  = f"explicit chars '{preview}'"

        base_cond = F.col(col).isNotNull()
        if not allow_empty:
            base_cond = base_cond & (F.length(F.col(col)) > 0)

        bad_count = df.filter(
            base_cond & ~F.col(col).cast("string").rlike(pattern)
        ).count()
        passed = bad_count == 0

        return DQRuleResult(
            **base, passed=passed, row_count=bad_count,
            details=(
                f"{bad_count} row(s) in '{col}' contain characters outside {pattern_desc}."
                if not passed else
                f"All non-null values in '{col}' conform to {pattern_desc}. ✓"
            ),
        )

    def _rule_length_check(self, rule, df, base):
        """
        Validates that the string length of a column falls within [min_length, max_length].
        Null values are always excluded.
        """
        col          = rule["column"]
        min_len      = rule.get("min_length")
        max_len      = rule.get("max_length")
        count_mode   = rule.get("count_mode", "chars").lower()
        strip_before = rule.get("strip_before_check", False)

        str_expr = F.col(col).cast("string")
        if strip_before:
            str_expr = F.trim(str_expr)

        len_expr   = F.octet_length(str_expr) if count_mode == "bytes" else F.length(str_expr)
        mode_label = count_mode

        cond = F.lit(False)
        if min_len is not None:
            cond = cond | (len_expr < int(min_len))
        if max_len is not None:
            cond = cond | (len_expr > int(max_len))

        bad_count = df.filter(F.col(col).isNotNull() & cond).count()
        passed    = bad_count == 0

        # On failure, surface actual data min/max for diagnosis
        extra = ""
        if not passed:
            stats = (
                df.filter(F.col(col).isNotNull())
                  .agg(F.min(len_expr).alias("mn"), F.max(len_expr).alias("mx"))
                  .collect()[0]
            )
            extra = f" Actual data range: [{stats['mn']}, {stats['mx']}] {mode_label}."

        lo   = str(min_len) if min_len is not None else "∞"
        hi   = str(max_len) if max_len is not None else "∞"
        note = " (whitespace stripped)" if strip_before else ""

        return DQRuleResult(
            **base, passed=passed, row_count=bad_count,
            details=(
                f"{bad_count} row(s) in '{col}' have length outside [{lo}, {hi}] {mode_label}{note}.{extra}"
                if not passed else
                f"All non-null '{col}' values have length within [{lo}, {hi}] {mode_label}{note}. ✓"
            ),
        )

    # ─────────────────────────────────────────────────────────────────────────
    # Private helpers for conditional_not_null
    # ─────────────────────────────────────────────────────────────────────────

    @staticmethod
    def _build_condition(cond_col, op, cond_val):
        """
        Build a Spark Column expression for the conditional_not_null trigger.
        Supported operators: eq | ne | gt | gte | lt | lte |
                             in | not_in | is_null | is_not_null
        """
        c = F.col(cond_col)
        if op == "eq":
            return c == cond_val
        elif op == "ne":
            return c != cond_val
        elif op == "gt":
            return c > cond_val
        elif op == "gte":
            return c >= cond_val
        elif op == "lt":
            return c < cond_val
        elif op == "lte":
            return c <= cond_val
        elif op == "in":
            vals = cond_val if isinstance(cond_val, list) else [cond_val]
            return c.isin(vals)
        elif op == "not_in":
            vals = cond_val if isinstance(cond_val, list) else [cond_val]
            return ~c.isin(vals)
        elif op == "is_null":
            return c.isNull()
        elif op == "is_not_null":
            return c.isNotNull()
        else:
            raise ValueError(f"Unknown condition_operator '{op}'.")

    @staticmethod
    def _op_display(op, cond_col, cond_val):
        """Human-readable description of a condition for result details."""
        if op in ("is_null", "is_not_null"):
            return f"{cond_col} {op.replace('_', ' ').upper()}"
        return f"{cond_col} {op.upper()} {cond_val!r}"

### ✅ Rule engine notebook loaded

Defines: `DQRuleResult`, `RuleEngine` (16 rule types)
Available after `%run` in the calling notebook.